# Assignment #03

by Patrick Donnelly & Burke Havranek

EECE 5644: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Scenario

### **1. Company Background**

WheelsBazaar is an online used-car marketplace operating across India (similar to Cars24, CarDekho, or
Spinny). Sellers list their vehicles on our platform, and buyers browse, compare, and purchase. We currently
handle roughly 5,000 listings per month across metros and tier-2 cities.

### **2. Business Problem**
Today, sellers set their own asking price, and our operations team manually reviews listings that "look off." This
causes three measurable problems:
1. **Overpriced listings sit unsold.** Cars priced more than ~15% above market value take 3x longer to sell,
clogging inventory and hurting buyer trust in the platform.
2. **Underpriced listings cost sellers money** — and generate complaints when sellers later discover they
sold below market. This damages retention.
3. **Manual review doesn't scale.** Our pricing analysts can review ~40 listings a day. At 5,000
listings/month we review less than 25% of inventory.
We also plan to launch an instant-buy program (WheelsBazaar Direct) where the company purchases cars
directly from sellers. For this, we need an auto

## Part 1: Data Retrieval and Sanitization

In [ ]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

df = pd.read_csv("car_details_v4.csv")
df

Thus, we find 2059 entries with 20 individual fields. First, we examine any missing or malformed entries:

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

We find several across several fields with missing entries. To determine whether they are correlated, let us examine first the most deficient field:

In [ ]:
df[df["Drivetrain"].isnull()].head(10)

From this, we find that the spread of invalid entries tend to correlate quite strongly with each other, but not with any other fields in particular. We are unsure as to the relative importance of each field in determining price, however. Since these incomplete entries compose a rather small percentage of the total data set (~9%), and due to a lack of any defensible default value for the categorical value of `Drivetrain`, we have therefore chosen to drop incomplete entries.

In [ ]:
df.dropna(inplace=True)
df

Let us now consider the format of each entry for each field. For such entries as `Make` and `Model`, these fields are purely categorical, and cannot be meaningfully reduced further other than through encoding, as discussed below. However, for such fields as `Engine`, `Max Power`, and `Max Torque`, these fields carry continuous numerical information in addition to units, and may therefore uniformly sanitized as follows:

In [ ]:
# Given a string, extracts the first floating-point number
get_first_number = lambda x: float(re.search(r'^\d+\.?\d*', x).group())

df["Engine"] = df["Engine"].apply(get_first_number)
df["Max Power"] = df["Max Power"].apply(get_first_number)
df["Max Torque"] = df["Max Torque"].apply(get_first_number)

df

Next, we deal with the categorical data, beginning with the number of unique entries:

In [ ]:
df.nunique()

As expected, the categories of `Fuel Type`, `Transmission`, `Owner`, `Seller Type`, and `Drivetrain` have few entries. For `Color`, `Make`, and especially `Location` and `Model` have significantly more unique entries, the latter most comprising almost half unique entries. It is at this point that we must determine what features are significant to the cost of a vehicle.

While the `Model` of a car certainly affects its price, the sheer number of unique entries relative to the number of total entries makes any model training unlikely to converge upon a pattern, having only on average about two vehicles per unique model. We must therefore drop the `Model` field so as to not introduce undue noise to the model training.

The `Make` of a car, however, is significant to its price, with luxury brands commanding on average higher prices than budget brands. As such, this field must be kept.

The `Location` and `Color` of a car are both interesting pieces of categorical data, as it is unclear whether they might affect the car's final price. For now, we keep them, but may later drop them during model tuning.

In [ ]:
df.drop("Model", axis=1, inplace=True)
df

Let us finally verify the aforementioned categorical data:

In [ ]:
print(df["Make"].unique(),end='\n\n')
print(df["Fuel Type"].unique(),end='\n\n')
print(df["Transmission"].unique(),end='\n\n')
print(df["Location"].unique(),end='\n\n')
print(df["Color"].unique(),end='\n\n')
print(df["Owner"].unique(),end='\n\n')
print(df["Seller Type"].unique(),end='\n\n')
print(df["Drivetrain"].unique(),end='\n\n')

Everything looks correct, with no discernable typos save for `FuelType`, in which there does not appear to be a distinction between `CNG` and `CNG + CNG`, hence the two entries will be consolidated:

In [ ]:
df["Fuel Type"] = df["Fuel Type"].replace("CNG + CNG", "CNG")
df["Fuel Type"].unique()

Lastly, the `Year`, `Transmission`, and `Owner` fields may be made generally numerically rather than categorical, producing `Age`, `Manual`, and `Owners` fields, respectively. Dropping the categorical fields, we produce:

In [ ]:
df["Age"] = 2026 - df["Year"]
df.drop("Year", axis=1, inplace=True)

df["Manual"] = df["Transmission"].map({"Automatic": 1, "Manual": 0})
df.drop("Transmission", axis=1, inplace=True)

df["Owners"] = df["Owner"].map({"UnRegistered Car": 0, "First": 1, "Second": 2, "Third": 3})
df.drop("Owner", axis=1, inplace=True)

df

## Part 2: Model Creation

## Part 3: Model Evaluation